<a href="https://colab.research.google.com/github/smartypants85/SCUDEM-Hill-Model-RK4/blob/RK4---Basic/HillModelRK4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import math
import csv

# Corrected age-5 values (MRI moment arm: Wood et al. 2006; strength: Daloia 2018 fit with MRI moment arm)
SEX = "boys"   # "boys" or "girls"
CHILD = {
    "boys":  dict(h=0.2190, d_r=0.0149, l_ts=0.1699, F_max=736.5),
    "girls": dict(h=0.2176, d_r=0.0147, l_ts=0.1626, F_max=603.7),
}

P = dict(
    h=CHILD[SEX]["h"],          # elbow to biceps origin on humerus (m)
    d_r=CHILD[SEX]["d_r"],      # elbow to biceps insertion on forearm (m), from MRI
    d_g=0.19,      # elbow to prosthesis center of mass (m)
    L=0.25,        # elbow to load in hand (m)
    x_3=0.08,      # elbow to socket rim (m)
    F_p1=0.98,     # prosthesis weight (N)
    F_p2=15.0,     # load weight (N)
    F_max=CHILD[SEX]["F_max"],  # max isometric elbow flexor force (N), all flexors combined
    l_opt=0.07,    # optimal muscle fiber length (m)
    l_ts=CHILD[SEX]["l_ts"],    # tendon slack length (m), from rho fit to Wood torque-angle shape
    v_max=0.7,     # max shortening speed (m/s)
    A=0.25,        # Hill force-velocity constant
    g=9.81,        # gravity (m/s^2)
    a_d=0.2,       # extra activation during the lift (0 to 1)
    T_L=0.6,       # length of the activation burst (s)
    theta0_deg=-10.0,  # starting forearm angle (deg above horizontal)
    omega0=0.0,    # starting angular velocity (rad/s)
    dt=0.005,      # time step (s)
    n_steps=200,   # number of steps (sheet uses 200 rows)
    w=0.05,        # socket width (m)
    s=0.08,        # socket engagement length (m)
    p_tol=100000.0,  # tissue pressure tolerance (Pa)
    a_0=None,      # baseline activation; None = activation that holds the arm at theta0
)

# Moment of inertia about the elbow (point masses)
I = (P["F_p1"] * P["d_g"] ** 2 + P["F_p2"] * P["L"] ** 2) / P["g"]
W = P["F_p1"] * P["d_g"] + P["F_p2"] * P["L"]



def geometry(th):
    l_mt = math.sqrt(P["h"] ** 2 + P["d_r"] ** 2 - 2 * P["h"] * P["d_r"] * math.sin(th))
    sin_a = P["h"] * math.cos(th) / l_mt
    l_m = l_mt - P["l_ts"]
    f_L = math.exp(-((l_m / P["l_opt"] - 1) ** 2) / 0.45)
    return l_mt, sin_a, l_m, f_L


def hold_activation(th):
    l_mt, sin_a, l_m, f_L = geometry(th)
    F_needed = math.cos(th) * W / (P["d_r"] * sin_a)
    return F_needed / (P["F_max"] * f_L)


theta0 = math.radians(P["theta0_deg"])
a_0 = P["a_0"] if P["a_0"] is not None else hold_activation(theta0)


def activation(t):
    a = a_0 + (P["a_d"] * math.sin(math.pi * t / P["T_L"]) ** 2 if t < P["T_L"] else 0.0)
    return min(a, 1.0), a


def muscle(t, th, om):
    l_mt, sin_a, l_m, f_L = geometry(th)
    v = P["d_r"] * sin_a * om                       # shortening speed (m/s)
    if v < 0:
        f_V = 1.0                                   # lengthening: use 1 (first model)
    else:
        f_V = max(0.0, (P["v_max"] - v) / (P["v_max"] + v / P["A"]))
    a, a_raw = activation(t)
    F_m = P["F_max"] * a * f_L * f_V
    dwdt = (F_m * P["d_r"] * sin_a - math.cos(th) * W) / I
    return dict(l_mt=l_mt, sin_a=sin_a, l_m=l_m, v=v, f_L=f_L, f_V=f_V,
                a=a, a_raw=a_raw, F_m=F_m, dwdt=dwdt)


def rhs(t, th, om):
    return om, muscle(t, th, om)["dwdt"]


def outputs(t, th, om):
    m = muscle(t, th, om)
    al = m["dwdt"]
    R_x = m["F_m"] * P["d_r"] * math.cos(th) / m["l_mt"] - W / P["g"] * (al * math.sin(th) + om ** 2 * math.cos(th))
    R_y = (P["F_p1"] + P["F_p2"] - m["F_m"] * (P["h"] - P["d_r"] * math.sin(th)) / m["l_mt"]
           + W / P["g"] * (al * math.cos(th) - om ** 2 * math.sin(th)))
    R = math.hypot(R_x, R_y)
    x3 = P["x_3"]
    M_s = (math.cos(th) * (P["F_p1"] * (P["d_g"] - x3) + P["F_p2"] * (P["L"] - x3))
           + al / P["g"] * (P["F_p1"] * P["d_g"] * (P["d_g"] - x3) + P["F_p2"] * P["L"] * (P["L"] - x3)))
    p_max = 6 * M_s / (P["w"] * P["s"] ** 2)
    return dict(t=t, theta=th, omega=om, theta_deg=math.degrees(th), **m,
                R_x=R_x, R_y=R_y, R=R, M_socket=M_s, p_max=p_max)


# RK4 starts here

def rk4(dt=P["dt"], n=P["n_steps"]):
    t, th, om = 0.0, theta0, P["omega0"]
    rows = [outputs(t, th, om)]
    for _ in range(n):
        k1th, k1om = rhs(t, th, om)
        k2th, k2om = rhs(t + dt / 2, th + dt / 2 * k1th, om + dt / 2 * k1om)
        k3th, k3om = rhs(t + dt / 2, th + dt / 2 * k2th, om + dt / 2 * k2om)
        k4th, k4om = rhs(t + dt,     th + dt * k3th,     om + dt * k3om)
        th += dt / 6 * (k1th + 2 * k2th + 2 * k3th + k4th)
        om += dt / 6 * (k1om + 2 * k2om + 2 * k3om + k4om)
        t += dt
        rows.append(outputs(t, th, om))
    return rows


def summary(rows, name):
    peak = lambda k: max(r[k] for r in rows)
    print(f"\n{name}")
    print(f"  Final theta (deg)      {rows[-1]['theta_deg']:.2f}")
    print(f"  Peak F_m (N)           {peak('F_m'):.1f}")
    print(f"  Peak |R| (N)           {peak('R'):.1f}")
    print(f"  Peak M_socket (N*m)    {peak('M_socket'):.3f}")
    print(f"  Peak p_max (Pa)        {peak('p_max'):,.0f}")
    print(f"  Peak p_max / p_tol     {peak('p_max') / P['p_tol']:.2f}")
    print(f"  Peak activation a      {peak('a'):.3f}  (uncapped {peak('a_raw'):.3f})")


if __name__ == "__main__":
    print(f"I = {I:.5f} kg m^2,  a_0 = {a_0:.4f}")
    if a_0 + P["a_d"] > 1:
        print(f"WARNING: a_0 + a_d = {a_0 + P['a_d']:.3f} > 1. The lift needs more than 100% effort;"
              " activation is capped at 1. Lower F_p2 or raise F_max.")

    rk = rk4()
    summary(rk, "RK4, dt = %.3f s" % P["dt"])


    # Save the RK4 table (same columns as the sheet) to CSV
    cols = ["t", "theta", "omega", "theta_deg", "l_mt", "sin_a", "l_m", "v", "f_L", "f_V",
            "a", "F_m", "dwdt", "R_x", "R_y", "R", "M_socket", "p_max"]
    with open("rk4_results.csv", "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(cols)
        for r in rk:
            wr.writerow([f"{r[c]:.6g}" for c in cols])
    print("\nSaved rk4_results.csv")

I = 0.09917 kg m^2,  a_0 = 0.4201

RK4, dt = 0.005 s
  Final theta (deg)      27.51
  Peak F_m (N)           331.8
  Peak |R| (N)           312.0
  Peak M_socket (N*m)    3.295
  Peak p_max (Pa)        61,781
  Peak p_max / p_tol     0.62
  Peak activation a      0.620  (uncapped 0.620)

Saved rk4_results.csv
